# Análisis exploratorio de datos – OT Consulting (OTC)
**Objetivo del cuaderno:** importar los 10 archivos Excel que salen del ERP interno de OTC, perfilar su calidad y explorar el contenido
(personal, nómina, asignaciones a clientes, activos y habilidades) para sustentar el documento de análisis.

**Fecha de corte del análisis:** 17-nov-2026. Los datos personales sensibles (RFC, CURP, NSS, teléfono, correo, domicilio) vienen enmascarados en la fuente.

## 0. Importación de los archivos

In [ ]:
import os, warnings
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
pd.set_option('display.width', 220, 'display.max_columns', 60, 'display.max_colwidth', 45, 'display.max_rows', 120)

FECHA_CORTE = pd.Timestamp('2026-11-17')
HORAS_MES = 160            # supuesto: 40 h/semana x 4 semanas (regla de negocio R10)

ARCHIVOS = {
 'colab' : 'Catalogo_Colaboradores.xlsx',
 'sueldos': 'Sueldos_Salarios_OTC.xlsx',
 'asig'  : 'Asignaciones_Colaboradores.xlsx',
 'clientes': 'Catalogo_Clientes.xlsx',
 'activos': 'Catalogo_Activos.xlsx',
 'inversion': 'Inversion_Activos_OTC.xlsx',
 'roles' : 'Catalogo_Roles.xlsx',
 'deptos': 'Catalogo_Departamentos.xlsx',
 'habs'  : 'Catalogo_Habilidades.xlsx',
 'matriz': 'Matriz_Habilidades.xlsx',
}
# Si los archivos no están en /content se piden con el cuadro de carga de Colab
faltan = [f for f in ARCHIVOS.values() if not os.path.exists('/content/' + f)]
if faltan:
    from google.colab import files
    files.upload()
print('Archivos presentes en /content:')
for f in sorted(ARCHIVOS.values()):
    print(f'  {f:<34} {os.path.getsize("/content/"+f)/1024:8.1f} KB')

In [ ]:
# Lectura de cada Excel en un DataFrame
D = {k: pd.read_excel('/content/' + v) for k, v in ARCHIVOS.items()}
colab, sueldos, asig, clientes = D['colab'], D['sueldos'], D['asig'], D['clientes']
activos, inversion, roles, deptos, habs, matriz = D['activos'], D['inversion'], D['roles'], D['deptos'], D['habs'], D['matriz']

inventario = pd.DataFrame({
    'archivo': [ARCHIVOS[k] for k in D],
    'filas': [d.shape[0] for d in D.values()],
    'columnas': [d.shape[1] for d in D.values()],
    'celdas_nulas': [int(d.isna().sum().sum()) for d in D.values()],
    '%_nulo': [round(d.isna().mean().mean()*100, 1) for d in D.values()],
    'filas_duplicadas': [int(d.duplicated().sum()) for d in D.values()],
})
inventario

In [ ]:
# Vista previa y tipos de dato de cada tabla
for k, d in D.items():
    print('='*100); print(ARCHIVOS[k], d.shape)
    display(d.head(3))

In [ ]:
def perfil(d, nombre=''):
    """Perfil de columnas: tipo, nulos, % nulos, valores únicos y un ejemplo."""
    p = pd.DataFrame({'tipo': d.dtypes.astype(str), 'nulos': d.isna().sum(), '%nulos': (d.isna().mean()*100).round(1),
                      'unicos': d.nunique(), 'ejemplo': d.apply(lambda s: s.dropna().iloc[0] if s.notna().any() else None)})
    print('\n###', nombre, d.shape); return p
perfil(colab, 'Catalogo_Colaboradores')

In [ ]:
perfil(asig, 'Asignaciones_Colaboradores')

In [ ]:
perfil(clientes, 'Catalogo_Clientes')

In [ ]:
perfil(activos, 'Catalogo_Activos')

In [ ]:
for k in ['sueldos','inversion','roles','deptos','habs']:
    display(perfil(D[k], ARCHIVOS[k]))

## 1. Colaboradores: estatus, nómina, puestos y antigüedad

In [ ]:
colab['Fecha Nacimiento'] = pd.to_datetime(colab['Fecha Nacimiento'], errors='coerce')
colab['Fecha Incorporación'] = pd.to_datetime(colab['Fecha Incorporación'], errors='coerce')
colab['Fecha de Baja'] = pd.to_datetime(colab['Fecha de Baja'], errors='coerce')
colab['edad'] = ((FECHA_CORTE - colab['Fecha Nacimiento']).dt.days/365.25).round(1)
colab['antig_anios'] = ((colab['Fecha de Baja'].fillna(FECHA_CORTE) - colab['Fecha Incorporación']).dt.days/365.25).round(1)

print('Colaboradores por estatus'); display(colab['Estatus'].value_counts().to_frame('n').assign(pct=lambda x:(x.n/x.n.sum()*100).round(1)))
act = colab[colab.Estatus=='Activo'].copy(); baj = colab[colab.Estatus=='Baja'].copy()
print('Tipo de nómina (activos)'); display(act['Tipo Nómina'].value_counts(dropna=False).to_frame('n'))
print('Puestos (activos, top 12)'); display(act['Puesto'].value_counts().head(12).to_frame('n'))
print('Departamento (todos)'); display(colab['Departamento'].value_counts(dropna=False).to_frame('n'))
print('Edad y antigüedad (años), todos los colaboradores'); display(colab[['edad','antig_anios']].describe().round(1))
print('Bajas por mes'); display(baj['Fecha de Baja'].dt.to_period('M').value_counts().sort_index().to_frame('bajas'))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
colab['Estatus'].value_counts().plot.bar(ax=ax[0], color=['#2a7','#c55'], rot=0); ax[0].set_title('Colaboradores por estatus'); ax[0].set_ylabel('n')
act['Tipo Nómina'].fillna('(sin dato)').value_counts().sort_values().plot.barh(ax=ax[1], color='#468'); ax[1].set_title('Activos por tipo de nómina')
plt.tight_layout(); plt.show()

## 2. Nómina y costo por colaborador

In [ ]:
cols_s = ['Sueldo Neto IMSS','Sueldo Asimilados/Sindicato','Sueldo Complementos','Sueldo Neto Total','Comisión Nómina','Impuestos IMSS','Sueldo Total Integrado']
# Cuadre de fórmulas
colab['chk_neto']  = colab['Sueldo Neto IMSS'] + colab['Sueldo Asimilados/Sindicato'] + colab['Sueldo Complementos']
colab['chk_integ'] = colab['Sueldo Neto Total'] + colab['Comisión Nómina'] + colab['Impuestos IMSS']
print('Neto Total ≠ IMSS + Asimilados + Complementos:', int((abs(colab.chk_neto-colab['Sueldo Neto Total'])>1).sum()))
print('Total Integrado ≠ Neto Total + Comisión + Impuestos IMSS:', int((abs(colab.chk_integ-colab['Sueldo Total Integrado'])>1).sum()))

# Costo por hora = Sueldo Total Integrado / horas al mes
colab['costo_hora'] = (colab['Sueldo Total Integrado'] / HORAS_MES).round(2)
act = colab[colab.Estatus=='Activo'].copy()
print('\nSueldos de colaboradores ACTIVOS (MXN/mes)'); display(act[cols_s+['costo_hora']].describe().round(0))
print('Costo mensual total de la nómina activa (Total Integrado): $', f"{act['Sueldo Total Integrado'].sum():,.0f}")
print('\nCosto promedio por tipo de nómina (activos con sueldo > 0)')
display(act[act['Sueldo Total Integrado']>0].groupby('Tipo Nómina', dropna=False)['Sueldo Total Integrado'].agg(['count','mean','median','sum']).round(0).sort_values('sum', ascending=False))
print('Composición promedio del pago (activos con sueldo > 0)')
comp = act[act['Sueldo Total Integrado']>0][['Sueldo Neto IMSS','Sueldo Asimilados/Sindicato','Sueldo Complementos','Comisión Nómina','Impuestos IMSS']].mean().round(0); display(comp.to_frame('promedio MXN'))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
act[act['Sueldo Total Integrado']>0]['costo_hora'].plot.hist(bins=15, ax=ax[0], color='#468'); ax[0].set_title('Costo por hora (Total Integrado / 160 h) – activos'); ax[0].set_xlabel('MXN por hora')
act[act['Sueldo Total Integrado']>0].boxplot(column='Sueldo Total Integrado', by='Tipo Nómina', ax=ax[1], rot=60); ax[1].set_title('Sueldo Total Integrado por tipo de nómina'); plt.suptitle('')
plt.tight_layout(); plt.show()

In [ ]:
# Consistencia entre Sueldos_Salarios_OTC y el catálogo de colaboradores
mm = sueldos.merge(colab[['ID','Estatus','Sueldo Neto Total','Sueldo Total Integrado','Puesto']], on='ID', how='left', suffixes=('','_cat'))
print('Filas en Sueldos:', len(sueldos), '| IDs que no existen en el catálogo:', int(mm.Estatus_cat.isna().sum()))
print('Diferencias de Sueldo Neto Total vs catálogo:', int((abs(mm['Sueldo Neto Total']-mm['Sueldo Neto Total_cat'])>0.01).sum()))
print('Activos del catálogo que no aparecen en Sueldos:', sorted(set(act.ID)-set(sueldos.ID)))
print('Activos con sueldo = 0:'); display(act[act['Sueldo Total Integrado']==0][['ID','Nombre Completo','Puesto','Tipo Nómina']])
print('Bajas con sueldo > 0 en el catálogo:', int((colab[colab.Estatus=='Baja']['Sueldo Total Integrado']>0).sum()))

## 3. Documentación del expediente (RH)

In [ ]:
docs = [c for c in colab.columns if c.startswith(('Doc:','Aviso:','Formato:'))]
est_doc = pd.DataFrame({c: colab.loc[colab.Estatus=='Activo', c].value_counts() for c in docs}).T.fillna(0).astype(int)
est_doc['%_pendiente'] = (est_doc.get('Pendiente', 0) / 75 * 100).round(0)
est_doc.sort_values('%_pendiente', ascending=False)

In [ ]:
(est_doc['%_pendiente']).sort_values().plot.barh(figsize=(8,6), color='#c85'); plt.title('% de pendientes por documento (75 colaboradores activos)'); plt.xlabel('%'); plt.tight_layout(); plt.show()
act_docs = colab[colab.Estatus=='Activo'].copy()
core = ['Doc: Acta','Doc: INE','Doc: CURP','Doc: Const. Situación Fiscal','Doc: NSS','Doc: Comprobante Domicilio','Doc: Banco']
act_docs['docs_ok'] = (act_docs[core]=='SI').sum(axis=1)
print('Expedientes básicos completos (7 de 7):', int((act_docs.docs_ok==7).sum()), 'de', len(act_docs))
act_docs['docs_ok'].value_counts().sort_index().plot.bar(rot=0, color='#468'); plt.title('Documentos básicos completos por colaborador'); plt.show()

## 4. Asignaciones a clientes y proyectos

In [ ]:
asig['ini'] = pd.to_datetime(asig['Fecha Inicio'], errors='coerce')
asig['fin'] = pd.to_datetime(asig['Fecha Fin'], errors='coerce')
asig['dias_para_fin'] = (asig['fin'] - FECHA_CORTE).dt.days
print('Valores de Fecha Fin que no son fecha:', asig.loc[asig.fin.isna(),'Fecha Fin'].unique())
print('Fechas de inicio = 1990-01-01 (marcador):', int((asig['Fecha Inicio']=='1990-01-01').sum()), 'de', len(asig))
print('Rol Asignado = PENDIENTE:', int((asig['Rol Asignado']=='PENDIENTE').sum()), '| Nómina/Rol Interno = PENDIENTE:', int((asig['Nómina/Rol Interno']=='PENDIENTE').sum()))
print('\nAsignaciones por cliente / proyecto'); display(asig.groupby(['Cliente','Proyecto']).size().to_frame('personas'))
print('Tipo de asignación'); display(asig['Tipo de Asignación'].value_counts().to_frame('n'))
dup = asig['ID Colaborador'].value_counts(); print('Colaboradores con más de una asignación:'); display(asig[asig['ID Colaborador'].isin(dup[dup>1].index)])
n_act = int((colab.Estatus=='Activo').sum()); n_asig = asig['ID Colaborador'].nunique()
print(f'Activos con alguna asignación: {n_asig} de {n_act} ({n_asig/n_act:.0%}). Sin asignación registrada: {n_act-n_asig}')
print('\nAsignaciones con fecha fin conocida y días restantes'); display(asig[asig.fin.notna()][['ID Colaborador','Nombre Colaborador','Cliente','Proyecto','Fecha Inicio','Fecha Fin','dias_para_fin']].sort_values('dias_para_fin'))

In [ ]:
# Colaboradores sin asignación (banca potencial) por puesto
sin_asig = colab[(colab.Estatus=='Activo') & (~colab.ID.isin(asig['ID Colaborador']))]
print('Sin asignación registrada:', len(sin_asig)); display(sin_asig['Puesto'].value_counts().head(10).to_frame('n'))
display(sin_asig['Tipo Nómina'].value_counts(dropna=False).to_frame('n'))
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
asig.groupby('Cliente').size().sort_values().plot.barh(ax=ax[0], color='#468'); ax[0].set_title('Personas asignadas por cliente')
pd.Series({'Con asignación': n_asig, 'Sin asignación': n_act-n_asig}).plot.bar(ax=ax[1], rot=0, color=['#2a7','#c55']); ax[1].set_title('Cobertura de asignación (activos)')
plt.tight_layout(); plt.show()

In [ ]:
# Cruce Asignaciones vs Catálogo de Clientes/Proyectos (fechas y existencia del proyecto)
clientes['ini_p'] = pd.to_datetime(clientes['Fecha Inicio'], errors='coerce'); clientes['fin_p'] = pd.to_datetime(clientes['Fecha Fin'], errors='coerce')
cr = asig.merge(clientes[['Cliente','Proyecto','ini_p','fin_p','Estatus Proyecto','PM OTC']], on=['Cliente','Proyecto'], how='left')
print('Asignaciones cuyo proyecto no existe en el catálogo de clientes:', int(cr['Estatus Proyecto'].isna().sum()))
inc = cr[(cr.fin_p.notna()) & (cr.fin.notna()) & (cr.fin != cr.fin_p)].drop_duplicates(['Cliente','Proyecto','Fecha Fin'])
print('Proyectos cuya fecha fin difiere entre Asignaciones y Clientes:'); display(inc[['Cliente','Proyecto','Fecha Inicio','ini_p','Fecha Fin','fin_p','Estatus Proyecto']])
vencidos = clientes[(clientes.fin_p < FECHA_CORTE) & (clientes['Estatus Proyecto']=='En Curso')]
print('Proyectos "En Curso" con fecha fin ya vencida al corte:'); display(vencidos[['Cliente','Proyecto','Fecha Fin','Estatus Proyecto']])
proy_sin_personal = clientes.merge(asig[['Cliente','Proyecto']].drop_duplicates().assign(tiene=1), on=['Cliente','Proyecto'], how='left')
print('Proyectos del catálogo sin ninguna persona asignada:'); display(proy_sin_personal[proy_sin_personal.tiene.isna()][['Cliente','Proyecto','Estatus Proyecto']])

## 5. Clientes

In [ ]:
display(clientes.drop(columns=['ini_p','fin_p']))
print('Clientes:', clientes['Cód Cliente'].nunique(), '| Proyectos:', len(clientes))
print('Proyectos por estatus'); display(clientes['Estatus Proyecto'].value_counts().to_frame('n'))
print('Clientes por industria'); display(clientes.drop_duplicates('Cód Cliente')['Industria'].str.strip().value_counts().to_frame('n'))
print('Campos vacíos en proyectos:'); display((clientes.isna().mean()*100).round(0).to_frame('%nulo').T)

## 6. Activos (equipo de cómputo)

In [ ]:
activos['Marca_std'] = activos['Marca'].str.strip().str.upper()
print('Variantes de marca en origen:', activos['Marca'].nunique(), '→ estandarizadas:', activos['Marca_std'].nunique())
display(pd.DataFrame({'origen': activos['Marca'].value_counts(), }).T)
display(activos['Marca_std'].value_counts().to_frame('equipos'))
print('Inversión total (Catalogo_Activos):', f"{activos.Costo.sum():,.0f}", '| Inversion_Activos_OTC:', f"{inversion.Costo.sum():,.0f}", '| coinciden fila a fila:', bool((activos.Costo.values==inversion.Costo.values).all()))
print('Equipos con costo = 0:', int((activos.Costo==0).sum()), 'de', len(activos), f"({(activos.Costo==0).mean():.0%})")
con_costo = activos[activos.Costo>0]; print('Costo promedio de equipos con costo registrado:', round(con_costo.Costo.mean()), '| mediana:', con_costo.Costo.median())
print('Sin asignar:', int((activos['Colaborador Asignado']=='Sin Asignar').sum()))
print('Fecha Inicio = 1990-01-01 (marcador):', int((activos['Fecha Inicio']=='1990-01-01').sum()), '| Fecha Fin con valor "Activo":', int((activos['Fecha Fin']=='Activo').sum()))
est = colab.set_index('Nombre Completo')['Estatus']
activos['estatus_resp'] = activos['Colaborador Asignado'].map(est)
print('\nEquipos asignados a colaboradores en BAJA (por recuperar):'); display(activos[activos.estatus_resp=='Baja'][['Cód Activo','Marca_std','Modelo','Colaborador Asignado','Fecha Inicio','Fecha Fin']])
multi = activos[activos.estatus_resp=='Activo']['Colaborador Asignado'].value_counts(); print('Colaboradores activos con más de 1 equipo:'); display(multi[multi>1].to_frame('equipos'))
sin_equipo = colab[(colab.Estatus=='Activo') & (~colab['Nombre Completo'].isin(activos['Colaborador Asignado']))]
print('Colaboradores activos sin equipo asignado:', len(sin_equipo))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
activos['Marca_std'].value_counts().plot.bar(ax=ax[0], rot=0, color='#468'); ax[0].set_title('Equipos por marca (estandarizada)')
con_costo['Costo'].plot.hist(bins=12, ax=ax[1], color='#468'); ax[1].set_title('Distribución del costo (costo > 0)')
activos['estatus_resp'].fillna('Sin asignar').value_counts().plot.bar(ax=ax[2], rot=0, color=['#2a7','#999','#c55']); ax[2].set_title('Estatus del responsable del equipo')
plt.tight_layout(); plt.show()

## 7. Roles, departamentos y habilidades

In [ ]:
print('Roles por nivel'); display(roles['Nivel'].value_counts().to_frame('roles'))
print('Roles sin departamento:'); display(roles[roles.Departamento.isna()][['Código','Nombre del Rol','Nivel']])
print('Puestos de colaboradores que no existen en el catálogo de roles:', sorted(set(colab.Puesto)-set(roles['Nombre del Rol'])))
print('\nDepartamentos: campos responsable / centroCostos vacíos →', int(deptos.responsable.isna().sum()), '/', int(deptos.centroCostos.isna().sum()), 'de', len(deptos))
display(deptos[['codigo','nombre','objetivo']])
print('Catálogo de habilidades por tecnología y tipo'); display(pd.crosstab(habs['tecnologia'], habs['tipo']))

In [ ]:
col_h = [c for c in matriz.columns if ' - ' in c and c[0] in 'HS']
orden = {'Básico':1, 'Intermedio':2, 'Avanzado':3, 'Experto':4}
niv = matriz[col_h].replace(orden).apply(pd.to_numeric, errors='coerce')
tot = niv.size; print('Celdas persona×habilidad:', tot, '| con nivel:', int(niv.notna().sum().sum()), f"({niv.notna().sum().sum()/tot:.0%})", '| vacías:', int(niv.isna().sum().sum()))
print('Distribución de niveles'); display(pd.Series(matriz[col_h].values.ravel()).value_counts(dropna=False).rename_axis('nivel').to_frame('celdas'))
matriz['n_habs'] = niv.notna().sum(axis=1); matriz['nivel_prom'] = niv.mean(axis=1).round(2)
act_ids = set(colab.loc[colab.Estatus=='Activo','ID'])
print('Activos sin fila en la matriz:', len(act_ids-set(matriz['ID Colaborador'])), '| filas de la matriz con estatus Baja:', int((matriz.Estatus=='Baja').sum()))
print('\nHabilidades por persona'); display(matriz['n_habs'].describe().round(1).to_frame('n_habs').T)
print('Personas con ≤ 3 habilidades evaluadas (matriz poco confiable):'); display(matriz[matriz.n_habs<=3][['ID Colaborador','Nombre Completo','Estatus','n_habs']])
resumen = pd.DataFrame({'evaluados': niv.notna().sum(), 'expertos': (niv==4).sum(), 'avanzados+': (niv>=3).sum(), 'nivel_prom': niv.mean().round(2)}).sort_values('avanzados+', ascending=False)
print('Habilidades con más personas nivel Avanzado o Experto'); display(resumen.head(12))
print('Habilidades con 0 personas nivel Avanzado/Experto (huecos de cobertura):'); display(resumen[resumen['avanzados+']==0])

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
pd.Series(matriz[col_h].values.ravel()).value_counts().reindex(['Básico','Intermedio','Avanzado','Experto']).plot.bar(ax=ax[0], rot=0, color='#468'); ax[0].set_title('Distribución de niveles de habilidad')
resumen['avanzados+'].head(12).sort_values().plot.barh(ax=ax[1], color='#2a7'); ax[1].set_title('Top 12 habilidades con más Avanzado/Experto')
plt.tight_layout(); plt.show()

## 8. Bitácora de errores, faltantes e inconsistencias (modelo sugerido)

In [ ]:
H = []
def add(tabla, campo, tipo, descripcion, n, total, sev, accion):
    H.append(dict(tabla=tabla, campo=campo, tipo=tipo, descripcion=descripcion, afectados=int(n), universo=int(total), pct=round(100*n/total,1) if total else None, severidad=sev, accion_sugerida=accion))

act = colab[colab.Estatus=='Activo']; baj = colab[colab.Estatus=='Baja']
# --- Colaboradores
add('Colaboradores','Puesto','Dato faltante','Puesto = PENDIENTE (todos los colaboradores)', (colab.Puesto=='PENDIENTE').sum(), len(colab), 'Alta','Asignar puesto del catálogo de roles')
add('Colaboradores','Puesto','Dato faltante','Puesto = PENDIENTE en colaboradores ACTIVOS', (act.Puesto=='PENDIENTE').sum(), len(act), 'Alta','Completar antes de calcular costo por rol')
add('Colaboradores','Departamento','Dato faltante','Departamento vacío', colab.Departamento.isna().sum(), len(colab), 'Alta','Derivar del rol (catálogo de roles)')
add('Colaboradores','Tipo Nómina','Dato faltante','Tipo de nómina vacío', colab['Tipo Nómina'].isna().sum(), len(colab), 'Media','Capturar o marcar como bajas sin nómina')
add('Colaboradores','Tipo Nómina','Inconsistencia','Categorías solapadas: SMR / OTC-SINDICATO (SMR) / MODALIDAD 40-SMR', colab['Tipo Nómina'].str.contains('SMR', na=False).sum(), len(colab), 'Media','Estandarizar a una sola clave de esquema')
add('Colaboradores','Fecha de Baja','Inconsistencia','Estatus Baja sin fecha de baja', ((colab.Estatus=='Baja') & colab['Fecha de Baja'].isna()).sum(), len(baj), 'Alta','Recuperar fecha de baja')
add('Colaboradores','Fecha de Baja','Inconsistencia','Estatus Activo con fecha de baja registrada (baja futura 19-nov-2026)', ((colab.Estatus=='Activo') & colab['Fecha de Baja'].notna()).sum(), len(act), 'Media','Marcar como baja programada')
add('Colaboradores','Sueldo Total Integrado','Inconsistencia','Activos con sueldo = 0', (act['Sueldo Total Integrado']==0).sum(), len(act), 'Alta','Confirmar si son administrativos fuera de nómina o falta captura')
add('Colaboradores','Sueldo Total Integrado','Inconsistencia','Bajas que conservan sueldo > 0', (baj['Sueldo Total Integrado']>0).sum(), len(baj), 'Media','Poner en 0 al dar de baja o conservar histórico en otra tabla')
add('Colaboradores','Fecha Nacimiento / Incorporación','Dato faltante','Sin fecha de nacimiento ni de incorporación', (colab['Fecha Nacimiento'].isna() | colab['Fecha Incorporación'].isna()).sum(), len(colab), 'Baja','Capturar (registro FAMILIAR COLABORADOR)')
add('Colaboradores','Fecha Nacimiento','Valor atípico','Edad mayor a 70 años (nacimiento 1946)', (colab.edad>70).sum(), len(colab), 'Baja','Validar captura')
add('Colaboradores','SGMM','Inconsistencia','Valores mezclados (PLAN SEGURO/PLAN SEGUROS, SI/NO, PENDIENTE, NO LO ENVIO…)', colab.SGMM.nunique(), colab.SGMM.notna().sum(), 'Media','Catálogo de aseguradoras + bandera de cobertura')
add('Colaboradores','Ciudad','Inconsistencia','Ciudad sin estandarizar (MTY/Monterrey, Jalisco/Guadalajara, país Colombia, Matriz Coporativa…)', colab.Ciudad.nunique(), len(colab), 'Baja','Catálogo ciudad-estado-país')
add('Colaboradores','Beneficiario / Parentesco','Dato faltante','Beneficiario = Pendiente', (colab.Beneficiario=='Pendiente').sum(), len(colab), 'Media','Levantar beneficiarios (seguro de vida)')
add('Colaboradores','Notas','Columna vacía','Notas 100% vacía', colab.Notas.isna().sum(), len(colab), 'Baja','Eliminar o poblar')
for c_ in ['RFC','CURP','NSS','Teléfono','Correo Electrónico','Domicilio']:
    add('Colaboradores', c_, 'Dato enmascarado','Un solo valor para todos los registros (dato anonimizado, no sirve como identificador)', len(colab)-1 if colab[c_].nunique()==1 else 0, len(colab), 'Info','Usar ID como llave')
for c_ in ['Doc: Acta','Doc: INE','Doc: CURP','Doc: Const. Situación Fiscal','Doc: NSS','Doc: Comprobante Domicilio','Doc: Banco','Doc: Foto','Doc: Contrato','Formato: RH']:
    add('Colaboradores', c_, 'Dato faltante','Documento pendiente (activos)', (act[c_]=='Pendiente').sum(), len(act), 'Media' if c_!='Doc: Contrato' else 'Alta','Seguimiento de RH')
# --- Sueldos
add('Sueldos','Departamento','Dato faltante','Departamento vacío', sueldos.Departamento.isna().sum(), len(sueldos), 'Alta','Derivar del rol')
add('Sueldos','Puesto/Rol','Dato faltante','Puesto = PENDIENTE', (sueldos['Puesto/Rol']=='PENDIENTE').sum(), len(sueldos), 'Alta','Completar')
# --- Asignaciones
add('Asignaciones','Fecha Inicio','Valor marcador','Fecha inicio = 1990-01-01 (marcador de fecha desconocida)', (asig['Fecha Inicio']=='1990-01-01').sum(), len(asig), 'Alta','Recuperar fecha real')
add('Asignaciones','Fecha Fin','Tipo mezclado','Fecha fin = texto "Indefinido" en lugar de fecha', asig.fin.isna().sum(), len(asig), 'Media','Dejar nulo + bandera "indefinida"')
add('Asignaciones','Rol Asignado','Dato faltante','Rol asignado = PENDIENTE', (asig['Rol Asignado']=='PENDIENTE').sum(), len(asig), 'Alta','Capturar rol por proyecto')
add('Asignaciones','Nómina/Rol Interno','Dato faltante','Rol interno = PENDIENTE', (asig['Nómina/Rol Interno']=='PENDIENTE').sum(), len(asig), 'Media','Sincronizar con catálogo de colaboradores')
add('Asignaciones','Proyecto','Inconsistencia','"Asignación indefinida" funciona como proyecto comodín', (asig.Proyecto=='Asignación indefinida').sum(), len(asig), 'Alta','Reemplazar por proyecto/cliente real')
add('Asignaciones','Cobertura','Dato faltante','Activos sin asignación registrada', len(sin_asig), len(act), 'Alta','Confirmar si están en banca o falta captura')
add('Asignaciones','Fechas vs Clientes','Inconsistencia','Proyecto con fecha fin distinta entre Asignaciones y Catálogo de Clientes', len(inc), clientes['Proyecto'].nunique(), 'Alta','Definir fuente única de fechas de proyecto')
add('Asignaciones','Estatus Proyecto','Inconsistencia','Proyecto En Curso con fecha fin vencida', len(vencidos), len(clientes), 'Alta','Cerrar o renovar proyecto')
# --- Clientes
add('Clientes','RFC','Dato faltante','RFC vacío', clientes.RFC.isna().sum(), len(clientes), 'Media','Capturar')
add('Clientes','PM OTC','Dato faltante','Proyecto sin PM de OTC', clientes['PM OTC'].isna().sum(), len(clientes), 'Alta','Asignar PM')
add('Clientes','Fechas del proyecto','Dato faltante','Proyecto sin fecha de inicio', clientes['Fecha Inicio'].isna().sum(), len(clientes), 'Alta','Capturar')
add('Clientes','Fecha Fin','Dato faltante','Proyecto sin fecha de fin', clientes['Fecha Fin'].isna().sum(), len(clientes), 'Media','Capturar o marcar recurrente')
add('Clientes','Proyecto','Error de captura','Nombres con errores ("Manufacta", "Asiganci´ñon")', clientes.Proyecto.str.contains('Manufacta|Asiganci', regex=True).sum(), len(clientes), 'Baja','Corregir texto')
add('Clientes','Proyecto','Cobertura','Proyectos sin ninguna persona asignada', int(proy_sin_personal.tiene.isna().sum()), len(clientes), 'Media','Validar si son proyectos activos')
# --- Activos
add('Activos','Marca','Inconsistencia','Variantes de escritura de la misma marca', activos.Marca.nunique()-activos.Marca_std.nunique(), activos.Marca.nunique(), 'Baja','TRIM + MAYÚSCULAS')
add('Activos','Costo','Dato faltante','Costo = 0', (activos.Costo==0).sum(), len(activos), 'Alta','Buscar factura/proveedor')
add('Activos','Fecha Inicio','Valor marcador','Fecha inicio = 1990-01-01 o vacía', ((activos['Fecha Inicio']=='1990-01-01') | activos['Fecha Inicio'].isna()).sum(), len(activos), 'Media','Capturar fecha de entrega')
add('Activos','Fecha Fin','Tipo mezclado','Fecha fin con texto "Activo"', (activos['Fecha Fin']=='Activo').sum(), len(activos), 'Media','Dejar nulo + bandera vigente')
add('Activos','Colaborador Asignado','Inconsistencia','Equipo asignado a un colaborador en BAJA', (activos.estatus_resp=='Baja').sum(), len(activos), 'Alta','Recuperar equipo / reasignar')
add('Activos','Serie / Modelo','Dato faltante','Serie = PENDIENTE o modelo vacío', ((activos.Serie.str.upper()=='PENDIENTE') | activos.Modelo.isna()).sum(), len(activos), 'Baja','Completar')
add('Activos','Colaborador Asignado','Cobertura','Activos sin equipo asignado', len(sin_equipo), len(act), 'Media','Validar si usan equipo propio')
# --- Roles / Deptos / Matriz
add('Roles','Departamento','Dato faltante','Roles sin departamento', roles.Departamento.isna().sum(), len(roles), 'Media','Asignar departamento')
add('Departamentos','responsable / centroCostos','Columna vacía','Sin responsable ni centro de costos', len(deptos), len(deptos), 'Alta','Definir para poder costear por área')
add('Matriz','Celdas persona×habilidad','Dato faltante','Celdas sin nivel (no se distingue "no sabe" de "no evaluado")', int(niv.isna().sum().sum()), niv.size, 'Alta','Agregar nivel "Sin conocimiento" y fecha de evaluación')
add('Matriz','ID Colaborador','Cobertura','Activos sin fila en la matriz de habilidades', len(act_ids-set(matriz['ID Colaborador'])), len(act_ids), 'Alta','Levantar matriz')
add('Matriz','Estatus','Inconsistencia','Filas de colaboradores en baja', (matriz.Estatus=='Baja').sum(), len(matriz), 'Baja','Conservar sólo con fecha de corte')
add('Matriz','Departamento','Dato faltante','Departamento vacío', matriz.Departamento.isna().sum(), len(matriz), 'Media','Derivar del rol')
hallazgos = pd.DataFrame(H); hallazgos.insert(0, 'id', ['E%02d' % i for i in range(1, len(hallazgos)+1)])
hallazgos.to_csv('/content/hallazgos_calidad.csv', index=False, encoding='utf-8-sig')
print('Hallazgos registrados:', len(hallazgos)); display(hallazgos['severidad'].value_counts().to_frame('n'))
hallazgos

## 9. Métricas derivadas (primer cálculo)

In [ ]:
m = {}
act = colab[colab.Estatus=='Activo']
m['Headcount activo'] = len(act)
m['Bajas 2026 (con fecha)'] = int(baj['Fecha de Baja'].notna().sum())
m['Tasa de baja sobre total histórico'] = f"{len(baj)/len(colab):.1%}"
m['Costo mensual nómina activa (Total Integrado)'] = f"${act['Sueldo Total Integrado'].sum():,.0f}"
m['Costo promedio por hora (activos con sueldo)'] = f"${act[act['Sueldo Total Integrado']>0]['costo_hora'].mean():,.1f}"
m['Cobertura de asignación'] = f"{n_asig/n_act:.0%}"
m['Equipos registrados / inversión'] = f"{len(activos)} / ${activos.Costo.sum():,.0f}"
m['Equipos en manos de bajas'] = int((activos.estatus_resp=='Baja').sum())
m['Habilidades evaluadas (celdas)'] = f"{niv.notna().sum().sum()/niv.size:.0%}"
pd.Series(m).to_frame('valor')

In [ ]:
# Costo por hora de cada persona asignada a un proyecto y costo mensual por cliente (si todo su tiempo es al cliente)
cp = asig.merge(colab[['ID','Sueldo Total Integrado','costo_hora']], left_on='ID Colaborador', right_on='ID', how='left')
cp['n_asig'] = cp.groupby('ID Colaborador')['ID Colaborador'].transform('count')
cp['costo_mensual_asignado'] = cp['Sueldo Total Integrado'] / cp['n_asig']   # regla R6: si hay 2 asignaciones se reparte 50/50
print('Costo mensual de la plantilla asignada por cliente/proyecto (MXN)')
display(cp.groupby(['Cliente','Proyecto']).agg(personas=('ID Colaborador','nunique'), costo_mensual=('costo_mensual_asignado','sum'), costo_hora_prom=('costo_hora','mean')).round(0))

In [ ]:
# Exporta evidencia
colab.drop(columns=['chk_neto','chk_integ']).to_csv('/content/colaboradores_limpios_parcial.csv', index=False, encoding='utf-8-sig')
print('Archivos generados:', [f for f in os.listdir('/content') if f.endswith('.csv')])